# Criteo Uplift v2.1: теория uplift-моделирования и план работ на трёх человек

**Что это за ноутбук.** Здесь только markdown: теория, формулы, разбор датасета и распределение задач. Кода нет — каждый участник пишет свой пайплайн самостоятельно на **одинаковом разбиении 60 / 20 / 20** с фиксированным `random_state`.

**Для кого.** Три человека с примерно одинаковым уровнем: scikit-learn, pandas, numpy, matplotlib, базовые PyTorch/TensorFlow, деревья, бустинги, логистическая регрессия, элементарный feature engineering.

**Как читать.** Сначала все читают разделы 1–6 — это общая база. Затем каждый забирает свой трек из раздела 7. В конце — раздел 8: как честно сравнить результаты.

## Содержание

1. Бизнес-задача и датасет Criteo Uplift v2.1
   1.1. Какую бизнес-задачу решает датасет
   1.2. Четыре бизнес-сегмента
   1.3. Датасет Criteo Uplift Prediction Dataset v2.1
   1.4. **Глубокий разбор колонок и терминов (ОЧЕНЬ ВАЖНО)**
2. Причинно-следственная постановка: potential outcomes, CATE, ITE, ATE
3. Почему обычный ML (AUC, accuracy) здесь не работает
4. Метрики uplift: Qini, AUUC, Uplift@k, Normalized Qini
5. Семейства алгоритмов uplift
   5.1. Двухмодельный подход (T-learner)
   5.2. S-learner
   5.3. X-learner
   5.4. R-learner
   5.5. Class Variable Transformation (CVT)
   5.6. Uplift-деревья и леса (Uplift Random Forest, Causal Forest)
   5.7. Нейросетевые подходы (TARNet, DragonNet) — обзорно
6. Практические ловушки и правила гигиены
7. Разбиение на трёх человек: три равных трека
8. Как сравнивать результаты и что сдавать
9. Полезные библиотеки и источники

# 1. Бизнес-задача и датасет Criteo Uplift v2.1

## 1.1. Какую бизнес-задачу решает датасет

Представь рекламную кампанию. У компании есть бюджет, чтобы показать рекламу части пользователей. Обычный прогноз отклика отвечает на вопрос:

> «Кто вероятнее всего совершит покупку / визит?»

Но это **неправильный вопрос** для управления бюджетом. Потому что часть пользователей купила бы и без рекламы. Если мы покажем рекламу им — мы просто потратили деньги впустую. Есть пользователи, которые не купят никогда, — им реклама тоже не поможет. И есть небольшая группа, которая **купит именно потому, что увидела рекламу**. Вот на неё и нужно тратить бюджет.

Uplift-моделирование (другие названия: **incremental response modeling**, **true lift modeling**, **causal targeting**) отвечает на вопрос:

> «На кого реклама окажет **причинное дополнительное** воздействие?»

То есть нас интересует не `P(Y=1 | X)`, а разность:

$$ \tau(X) = P(Y=1 \mid T=1, X) - P(Y=1 \mid T=0, X) $$

где `T=1` — пользователь получил воздействие (treatment), `T=0` — контроль (control), `Y=1` — целевое событие (визит или конверсия).

## 1.2. Четыре бизнес-сегмента (классическая таксономия uplift)

По знаку и величине причинного эффекта пользователей делят на четыре типа:

| Сегмент | Что происходит | `P(Y\|T=1)` vs `P(Y\|T=0)` | Что делать |
|---|---|---|---|
| **Persuadables** (убеждаемые) | Покупают только под воздействием | `P1 > P0` | **Целевая аудитория**, показывать рекламу |
| **Sure things** (гарантированные) | Купят в любом случае | `P1 ≈ P0`, оба высокие | Не тратить деньги |
| **Lost causes** (потерянные) | Не купят ни при каком раскладе | `P1 ≈ P0`, оба низкие | Не тратить деньги |
| **Sleeping dogs** (спящие собаки) | Воздействие **снижает** вероятность | `P1 < P0` | **Избегать**, иначе вред |

Именно поэтому uplift-модель должна уметь давать **отрицательные** предсказания. Если модель обучена только предсказывать «вероятность отклика», она физически не может отделить `sleeping dogs` от `lost causes`.

## 1.3. Датасет Criteo Uplift Prediction Dataset v2.1

Датасет опубликован Criteo AI Lab. Это данные нескольких **рандомизированных контролируемых испытаний (RCT, randomized controlled trial)** в рекламе: случайная часть пользователей была лишена таргетинга (control), остальные могли быть обработаны (treatment).

**Размер:** около 13.9–14 млн строк (в v2.1; в ранней версии ~25 млн). Каждая строка — пользователь.

**Колонки:**

| Колонка | Тип | Смысл |
|---|---|---|
| `f0 … f11` | float | 12 анонимизированных признаков. Имена и значения перемешаны/хешированы так, чтобы сохранить предсказательную силу, но скрыть бизнес-смысл. |
| `treatment` | binary | 1 — пользователь в treatment-группе, 0 — в control. В датасете около 85% treatment. |
| `exposure` | binary | 1, если пользователь **фактически** увидел рекламу (эффективная экспозиция). Может не совпадать с `treatment`. |
| `visit` | binary | 1, если пользователь совершил визит. |
| `conversion` | binary | 1, если пользователь совершил конверсию (покупку / целевое действие). |

**Дисбаланс классов.** Базовые ставки низкие: `visit` порядка нескольких процентов, `conversion` — доли процента. Это значит, что **accuracy бесполезна**. Нужны метрики, чувствительные к ранжированию uplift.

## 1.4. Глубокий разбор колонок и терминов (ОЧЕНЬ ВАЖНО)

Разберём каждую колонку максимально подробно. В causal inference (причинном выводе) термины пришли из медицины, где тестируют лекарства. В рекламе всё аналогично: есть "лекарство" (реклама) и "плацебо" (отсутствие рекламы).

### 1. `treatment` (Воздействие / Лечение)

*   **Перевод:** Treatment — лечение, воздействие, обработка.
*   **Что это:** Это бинарный флаг (0 или 1). Он показывает, в какую **экспериментальную группу** был назначен пользователь.
    *   `treatment = 1`: Пользователь **назначен** на получение рекламы (в медицине — получил лекарство).
    *   `treatment = 0`: Пользователь **назначен** в контрольную группу (в медицине — получил плацебо).
*   **Важно:** В датасете Criteo примерно 85% строк имеют `treatment = 1` и только 15% — `treatment = 0`. Это дисбаланс, о котором нужно помнить при обучении моделей.
*   **Связь с бизнесом:** Мы не можем просто взять и отключить рекламу для половины пользователей ради эксперимента (это потеря денег). Поэтому в реальной жизни такие RCT (Randomized Controlled Trials) проводятся редко и очень аккуратно, а датасеты вроде Criteo — это "золотой стандарт", где такой эксперимент уже провели.

### 2. `control` (Контрольная группа)

*   **Перевод:** Control — контроль, управление.
*   **Что это:** Это НЕ отдельная колонка в датасете. Это **логическое название** для всех строк, где `treatment = 0`. 
*   **Зачем нужна:** Это "базовая линия". Мы смотрим на конверсии в контрольной группе, чтобы понять: "А что было бы, если бы мы вообще не показывали рекламу?". Без контрольной группы мы не сможем вычислить uplift (причинный эффект), потому что не с чем будет сравнивать.

### 3. `exposure` (Фактическая экспозиция / Показ)

*   **Перевод:** Exposure — подвергание воздействию, экспозиция, показ.
*   **Что это:** Бинарный флаг (0 или 1). Показывает, **фактически ли пользователь увидел рекламу**. 
*   **В чём подвох (и разница с treatment):** 
    *   Мы можем *назначить* пользователю рекламу (`treatment = 1`), но он мог пролистать ленту слишком быстро, у него мог заблокироваться баннер (AdBlock), или страница не догрузилась. В этом случае `exposure = 0`.
    *   Если пользователь увидел рекламу, то `exposure = 1`.
    *   В контрольной группе (`treatment = 0`) экспозиция по определению должна быть 0 (мы им ничего не показывали).
*   **Зачем это знать:** В базовом uplift-анализе мы работаем с `treatment`, а не с `exposure`. Это называется **Intention-to-Treat (ITT)** — анализ по назначенному лечению. Если использовать `exposure` как признак, можно получить **утечку данных (data leakage)**, потому что сам факт просмотра рекламы уже сильно коррелирует с покупкой.

### 4. `visit` (Визит)

*   **Перевод:** Visit — посещение, визит.
*   **Что это:** Бинарный таргет (0 или 1). 
    *   `1`: Пользователь кликнул на рекламу (или увидел её) и **зашёл на сайт**.
    *   `0`: Не зашёл.
*   **Связь с бизнесом:** Это "мягкая" конверсия. Визит случается чаще, чем покупка. В Criteo `visit` используется как прокси-метрика ценности (потому что клик стоит денег, а визит — это шаг к покупке).

### 5. `conversion` (Конверсия)

*   **Перевод:** Conversion — превращение, конверсия.
*   **Что это:** Бинарный таргет (0 или 1).
    *   `1`: Пользователь **совершил целевое действие** — купил товар, оформил подписку, оставил заявку.
    *   `0`: Не совершил.
*   **Связь с бизнесом:** Это "твёрдая" конверсия, самая важная для бизнеса. Но она случается крайне редко (доли процента), поэтому обучать на ней uplift-модели сложнее из-за сильного дисбаланса классов.

### 6. `f0, f1, ..., f11` (Анонимизированные признаки)

*   **Перевод:** Features — признаки, характеристики.
*   **Что это:** 12 колонок с числами. Это **фичи пользователя** (например, возраст, пол, доход, интересы, тип устройства, браузер, история покупок).
*   **Почему они называются `f0-f11`:** Criteo — это крупная рекламная сеть. Они не могут разглашать персональные данные пользователей (GDPR, приватность). Поэтому они **анонимизировали** признаки: убрали названия, а значения захешировали (превратили в числа).
*   **Как с ними работать (важный лайфхак):** Среди них есть как **вещественные (числовые)** признаки, так и **категориальные** (строки/ID, закодированные числами). 
    *   *Категориальные* — это те, где мало уникальных значений, или где числа не имеют математического смысла (например, `f1` может быть "город = 5", а не "возраст = 5"). Для них нужно делать OHE (One-Hot Encoding) или Target Encoding.
    *   *Числовые* — это те, где числа имеют смысл (например, время на сайте в секундах). Их можно масштабировать (StandardScaler) и использовать в линейных моделях.
    *   В EDA (разведочном анализе) вы обязательно должны посмотреть на уникальные значения (`.nunique()`) и распределения этих фич, чтобы понять, какие из них категориальные.

### Итоговая схема данных:
[ f0, f1, ..., f11 ] ---> Признаки пользователя (Кто он?)
|
v
[ treatment ] ---> Назначили ли ему рекламу? (1 = да, 0 = нет)
|
v
[ exposure ] ---> Увидел ли он её в реальности? (1 = да, 0 = нет)
|
v
[ visit, conversion ] ---> Что он сделал? (Зашёл на сайт? Купил?)

# 2. Причинно-следственная постановка

## 2.1. Potential outcomes (потенциальные исходы)

Для каждого пользователя `i` представь два **потенциальных** исхода:

- $Y_i(1)$ — что произошло бы, если бы пользователь получил воздействие;
- $Y_i(0)$ — что произошло бы, если бы **не** получил.

Индивидуальный причинный эффект:

$$ \tau_i = Y_i(1) - Y_i(0) $$

**Фундаментальная проблема причинности:** мы наблюдаем только одну из двух реалий. Если пользователь в treatment, мы видим $Y_i(1)$, но не знаем $Y_i(0)$; если в control — наоборот. Поэтому $\tau_i$ нельзя вычислить напрямую. Можно оценить **средний** эффект в группе с похожими признаками:

$$ \tau(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x] = \mathbb{E}[Y \mid T=1, X=x] - \mathbb{E}[Y \mid T=0, X=x] $$

Это и есть **CATE**. В RCT последнее равенство выполняется, потому что `T` не зависит от потенциальных исходов.

## 2.2. Допущения, которые мы принимаем

1. **SUTVA (Stable Unit Treatment Value Assumption).** Эффект воздействия на одного пользователя не зависит от того, какие воздействия получили другие. В рекламном RCT это разумно, если нет сильных сетевых эффектов.
2. **Unconfoundedness (ignorability).** При заданных `X` назначение treatment не зависит от потенциальных исходов. В RCT выполнено по дизайну.
3. **Positivity / overlap.** Для каждого $x$ есть ненулевая вероятность попасть и в treatment, и в control. В Criteo treatment ~85%, контроль ~15%, поэтому overlap есть, но по некоторым регионам признакового пространства может быть тонким.

Если эти допущения выполнены, задача сводится к оценке разности двух условных математических ожиданий.

## 2.3. ATE vs CATE vs ITE

- **ATE** — одно число: средний эффект по всем пользователям.
- **CATE** — функция от `X`: средний эффект для подгруппы с признаками $x$.
- **ITE** — предсказание эффекта для конкретного пользователя. На практике мы не можем проверить ITE на реальных данных (контрфактический исход неизвестен), но можем ранжировать пользователей по предсказанному CATE.

**Для бизнеса важно именно ранжирование.** Нам не нужна абсолютно точная величина $\tau(x)$; нам нужно правильно отсортировать пользователей, чтобы сверху оказались `persuadables`.

# 3. Почему обычный ML (AUC, accuracy) здесь не работает

## 3.1. Response-модель предсказывает не то

Обычный классификатор обучают на $P(Y=1 | X)$, игнорируя `T`. Он найдёт пользователей с высоким базовым откликом — это в основном **sure things** и **persuadables** вместе. Но `sure things` не нуждаются в рекламе, а мы не можем отличить их от `persuadables` без контрфактической информации.

**Response-модель (naive response ranker)** — важный baseline. Если uplift-модель не бьёт её по Qini, значит, она не научилась ничему причинному.

## 3.2. AUC ROC и accuracy нечувствительны к uplift

AUC ROC оценивает, насколько хорошо модель ранжирует **вероятность события**, а не **разность вероятностей**. Два пользователя могут иметь одинаковый `P(Y=1|X)`, но радикально разный $\tau(x)$. AUC этого не увидит.

Более того, при дисбалансе классов accuracy вырождается: константное предсказание нуля даёт >95% accuracy на `visit` и >99% на `conversion`.

## 3.3. Что использовать вместо этого

Нужны метрики, которые:
- явно сравнивают treatment и control;
- оценивают **накопленный инкрементальный** эффект при целевом отборе;
- устойчивы к низким базовым ставкам.

Это Qini, AUUC, Uplift@k. О них — следующий раздел.

# 4. Метрики uplift: Qini, AUUC, Uplift@k

## 4.1. Идея: сортируем по предсказанному uplift и считаем инкремент

Все uplift-метрики устроены одинаково:

1. Модель предсказывает $\hat{\tau}(x)$ для каждого пользователя.
2. Сортируем пользователей по убыванию $\hat{\tau}(x)$.
3. Берём верхнюю долю `t` (например, 10%, 20%, …, 100%).
4. В этой верхней доле считаем, сколько **дополнительных** положительных исходов дал treatment по сравнению с control.
5. Строим кривую «накопленный инкремент vs доля отобранных».
6. Считаем площадь под кривой.

## 4.2. Qini curve

Пусть:
- $N_T(t)$ — число treatment-пользователей в верхней доле `t`;
- $N_C(t)$ — число control-пользователей в верхней доле `t`;
- $R_T(t)$ — число положительных исходов среди treatment в верхней доле;
- $R_C(t)$ — число положительных исходов среди control в верхней доле.

**Qini curve:**

$$ Q(t) = R_T(t) - R_C(t) \cdot \frac{N_T(t)}{N_C(t)} $$

Множитель $N_T/N_C$ нужен, потому что treatment и control могут иметь разный размер (в Criteo treatment ~85%). Он масштабирует control к размеру treatment, чтобы сравнивать «как если бы группы были одинаковыми».

**Qini coefficient (нормализованный):**

$$ \text{Qini} = \frac{\text{AUUC}_{\text{model}} - \text{AUUC}_{\text{random}}}{\text{AUUC}_{\text{perfect}} - \text{AUUC}_{\text{random}}} $$

- $\text{AUUC}_{\text{model}}$ — площадь под Qini-кривой модели;
- $\text{AUUC}_{\text{random}}$ — площадь при случайном ранжировании;
- $\text{AUUC}_{\text{perfect}}$ — площадь, если бы мы знали истинный uplift (на реальных данных недостижимо, но используется для нормировки).

Qini = 1 — идеальное ранжирование; Qini = 0 — уровень случайности; Qini < 0 — модель хуже случайной.

## 4.3. AUUC (Area Under Uplift Curve)

AUUC — это площадь под кривой накопленного инкремента, но без нормировки на perfect/random. Иногда AUUC и Qini используют как синонимы; в индустрии чаще говорят «Qini» для нормализованной версии и «AUUC» для абсолютной.

**Uplift curve** часто определяют как:

$$ V(t) = R_T(t) - R_C(t)\frac{|T|}{|C|} $$

где $|T|$ и $|C|$ — общие размеры групп. Это кумулятивная версия Qini.

## 4.4. Uplift@k

Более простая и интерпретируемая метрика:

$$ \text{Uplift@k} = \frac{R_T(k)}{N_T(k)} - \frac{R_C(k)}{N_C(k)} $$

где `k` — верхние `k` пользователей (или верхние 10%, 20%). Это **разность конверсий** в верхней доле между treatment и control. Она показывает, насколько выше доля положительных исходов в treatment среди отобранных пользователей.

**Top-decile uplift** — Uplift@10% — удобен для отчёта бизнесу: «если мы покажем рекламу верхним 10% по версии модели, incremental visit rate составит X п.п.».

## 4.5. Важные предупреждения по метрикам

1. **Нельзя считать Qini на train.** Модель переобучится и покажет красивые числа. Qini считается только на validation и test.
2. **Нужен bootstrap для доверительных интервалов.** Qini — шумная метрика, особенно на маленьких выборках. Сравнивать модели по точечному Qini без CI — плохая практика. Рекомендуется bootstrap по пользователям (ресемплинг строк с возвращением) и построение 95% CI.
3. **Perfect Qini на реальных данных неизвестен.** Для нормировки можно использовать теоретическую верхнюю границу или просто сравнивать AUUC между моделями на одном и том же разбиении. Главное — **одинаковый split и одинаковая процедура оценки**.
4. **Qini может быть отрицательным.** Это нормально: модель может давать анти-ранжирование. Если Qini стабильно < 0 на validation, что-то не так с пайплайном (перепутаны группы, утечка, неверный знак).
5. **Сравнивать модели нужно на одном и том же наборе пользователей.** Если вы сравниваете Qini, посчитанный на разных подвыборках, сравнение нечестное.

# 5. Семейства алгоритмов uplift

Все методы ниже оценивают $\tau(x) = \mathbb{E}[Y(1)-Y(0)\mid X=x]$, но делают это по-разному. Мы разберём:

- мета-алгоритмы (T, S, X, R);
- преобразование целевой переменной (CVT);
- прямые uplift-деревья и леса;
- кратко — нейросетевые подходы.

## 5.1. Двухмодельный подход (T-learner, Two-Model)

**Идея.** Обучаем **две независимые** модели:

- $M_1(x) \approx P(Y=1 \mid T=1, X=x)$ — только на treatment-строках;
- $M_0(x) \approx P(Y=1 \mid T=0, X=x)$ — только на control-строках.

Предсказание uplift:

$$ \hat{\tau}(x) = M_1(x) - M_0(x) $$

**Плюсы:** просто, интерпретируемо, можно использовать любые базовые модели (логистическая регрессия, LightGBM, RF).

**Минусы:**
- Две модели обучаются на разных подвыборках, ошибки складываются.
- В control меньше данных (15%), поэтому $M_0$ может быть шумной.
- Модели не «знают» друг о друге, поэтому разность может быть нестабильной.

**Название «T-learner»** — от слова «two» (две модели) или «treatment/control».

**Практическая рекомендация для Criteo:** использовать T-learner как **первый baseline**. Он даёт точку отсчёта, ниже которой не должны опускаться более сложные методы.

## 5.2. S-learner (Single-model)

**Идея.** Обучаем **одну** модель на всех данных, подавая `treatment` как обычный признак:

$$ M(x, t) \approx P(Y=1 \mid X=x, T=t) $$

Предсказание uplift:

$$ \hat{\tau}(x) = M(x, 1) - M(x, 0) $$

**Плюсы:**
- Одна модель, проще и быстрее, чем T-learner.
- Модель видит все данные, поэтому оценки базовой вероятности стабильнее.
- Хорошо работает, если эффект treatment примерно однороден.

**Минусы:**
- Если базовый алгоритм «зажимает» признак `treatment` (например, дерево редко выбирает его для расщепления), модель может недооценить эффект.
- Сложнее уловить гетерогенность (разный эффект в разных подгруппах), потому что `T` — всего один бинарный признак среди многих.

**Название** — от «single».

## 5.3. X-learner

X-learner пытается исправить недостатки T-learner, особенно когда группы treatment и control **сильно разбалансированы** (как в Criteo, где treatment ~85%, control ~15%). Он состоит из **трёх этапов**.

**Этап 1.** Обучаем T-learner: две модели $M_1(x)$ и $M_0(x)$.

**Этап 2.** Вычисляем **imputed treatment effects** для каждого пользователя:

- Для treatment-пользователей:  
  $$ D_i^1 = Y_i - M_0(X_i) $$
  (фактический исход минус то, что предсказала бы control-модель).

- Для control-пользователей:  
  $$ D_i^0 = M_1(X_i) - Y_i $$
  (то, что предсказала бы treatment-модель, минус фактический исход).

Затем обучаем **две новые модели** на этих псевдоэффектах:

$$ \tau_1(x) \approx \mathbb{E}[D^1 \mid X=x], \quad \tau_0(x) \approx \mathbb{E}[D^0 \mid X=x] $$

**Этап 3.** Комбинируем их с весами:

$$ \hat{\tau}(x) = p(x)\tau_0(x) + (1-p(x))\tau_1(x) $$

где $p(x) = P(T=1 \mid X=x)$ — propensity score (вероятность попасть в treatment). В RCT $p(x)$ можно оценить как долю treatment в данных или обучить отдельный классификатор.

**Почему это работает:** если $M_0$ хорошо предсказывает control-исход, то $Y_i - M_0(X_i)$ для treatment-пользователя — это оценка эффекта. X-learner использует обе группы, чтобы восстановить эффект, и лучше справляется с дисбалансом.

**Плюсы:** устойчив к дисбалансу групп, часто даёт более стабильный uplift, чем T-learner.

**Минусы:** три этапа, больше гиперпараметров, сложнее в реализации и отладке.

## 5.4. R-learner (Robinson learner)

R-learner основан на **остатках** (residuals) и идее «residual-on-residual».

**Этап 1.** Обучаем две вспомогательные модели:

- $m(x) \approx \mathbb{E}[Y \mid X=x]$ — маргинальное ожидание исхода (без учёта `T`);
- $e(x) \approx P(T=1 \mid X=x)$ — propensity score.

**Этап 2.** Вычисляем остатки:

$$ \tilde{Y}_i = Y_i - m(X_i), \quad \tilde{T}_i = T_i - e(X_i) $$

**Этап 3.** Обучаем финальную модель на отношение остатков:

$$ \tau(x) \approx \mathbb{E}\left[ \frac{\tilde{Y}}{\tilde{T}} \,\middle|\, X=x \right] $$

но на практике минимизируют взвешенный квадратичный лосс:

$$ \min_{\tau} \sum_i \left( \tilde{Y}_i - \tau(X_i)\tilde{T}_i \right)^2 $$

Веса $\tilde{T}_i^2$ автоматически снижают вклад пользователей, у которых `T` почти детерминирован (propensity близко к 0 или 1).

**Плюсы:** эффективно использует все данные, устойчив к дисбалансу, хорошо работает с нелинейными базовыми моделями.

**Минусы:** требует оценки $m(x)$ и $e(x)$, чувствителен к переобучению вспомогательных моделей; нужно использовать cross-fitting.

## 5.5. Class Variable Transformation (CVT)

Идея — свести uplift к обычной бинарной классификации, **преобразовав целевую переменную**.

Определим новую переменную $Z$:

$$ Z_i = \begin{cases} 1, & \text{если } T_i = 1 \text{ и } Y_i = 1, \\ 1, & \text{если } T_i = 0 \text{ и } Y_i = 0, \\ 0, & \text{иначе.} \end{cases} $$

Тогда можно показать, что

$$ \tau(x) = 2 \cdot P(Z=1 \mid X=x) - 1 $$

То есть достаточно обучить **одну** модель $P(Z=1|X)$ и затем пересчитать в uplift. Это называется **class variable transformation** (CVT) или **modified outcome method**.

**Плюсы:** одна модель, можно использовать любой классификатор (логистическая регрессия, LightGBM), не нужно обучать две модели.

**Минусы:** меняется баланс классов (Z может быть сильно разбалансирован), интерпретация менее прямая. Тем не менее это отличный «дешёвый» baseline.

## 5.6. Uplift-деревья и леса

Обычное дерево решений выбирает расщепление, которое максимизирует чистоту по `Y`. Uplift-дерево выбирает расщепление, которое максимизирует **разницу uplift между дочерними узлами**.

**Критерий расщепления** (упрощённо): для каждого кандидата `A` считаем

$$ D_{\text{gain}}(A) = D\left(P(Y|T=1) : P(Y|T=0) \mid A\right) - D\left(P(Y|T=1) : P(Y|T=0)\right) $$

где $D$ — мера различия между распределениями treatment и control (например, KL-дивергенция, разность средних, критерий χ²).

**Uplift Random Forest** — ансамбль таких деревьев. Реализации:
- `causalml.inference.tree.UpliftRandomForestClassifier` (causalml);
- `econml` — Causal Forest (GRF), Honest Causal Tree.

**Causal Forest (GRF)** — это лес, который оценивает CATE с помощью «честного» разбиения (honest splitting): одна часть данных используется для выбора структуры дерева, другая — для оценки эффекта в листьях. Это даёт асимптотическую нормальность и возможность строить доверительные интервалы.

**Плюсы:** нелинейные взаимодействия, автоматический поиск гетерогенности, интерпретируемость через важность признаков.

**Минусы:** вычислительно тяжелее, чувствительны к гиперпараметрам, на 14M строк могут требовать сэмплирования.

## 5.7. Нейросетевые подходы (обзорно)

Для третьего участника, если захочется сложнее:

- **TARNet (Treatment-Agnostic Representation Network).** Общий «ствол» (representation) для `X`, затем две «головы» для treatment и control. Uplift = разность выходов голов. Обучение совместное, поэтому представление оптимизируется под обе группы.
- **DragonNet.** Добавляет propensity-голову и использует target regularization, чтобы представление было полезно и для исхода, и для treatment. Часто даёт более устойчивые оценки CATE.

На Criteo с 12 признаками нейросети не всегда бьют бустинги, но как **третий независимый подход** это интересно: он проверяет, не упускают ли деревья какую-то гладкую структуру.

# 6. Практические ловушки и правила гигиены

1. **Стратифицированное разбиение.** Делите данные на train/val/test **с учётом `treatment` и целевого исхода**. Иначе в одной из выборок может оказаться слишком мало положительных событий, и Qini будет шумным. Используйте `train_test_split(..., stratify=df[['treatment', 'visit']])`.

2. **Одинаковый split у всех трёх участников.** Зафиксируйте `random_state` и **порядок сортировки** перед разбиением. Лучше сохранить индексы train/val/test в отдельный CSV и передать всем. Иначе сравнение Qini между участниками будет нечестным.

3. **Не используйте `exposure` как признак или таргет без осознания.** `exposure` — это пост-обработка (фактическая экспозиция). Если использовать её как признак, можно получить утечку: exposure коррелирует с исходом и с treatment. Для базового uplift-анализа берите `treatment` как treatment-переменную, а `exposure` оставьте для отдельного анализа.

4. **Дисбаланс treatment (85/15).** T-learner обучит control-модель на маленькой выборке. Это нормально, но нужно проверять, что в control достаточно положительных событий. Если нет — сэмплируйте больше данных или используйте методы, устойчивые к дисбалансу (X-learner, R-learner).

5. **Утечка через ID или порядок строк.** Убедитесь, что в данных нет скрытого индекса, который коррелирует с таргетом. Перемешайте строки перед split.

6. **Калибровка вероятностей.** T-learner и S-learner дают вероятности, но для uplift важна не калибровка каждой модели по отдельности, а **калибровка разности**. После обучения можно проверить calibration plot для $\tau(x)$ (по децилям предсказанного uplift vs фактический uplift). Если модель переуверена, Qini может страдать.

7. **Сэмплирование.** 14M строк — это много для ноутбука. Можно взять стратифицированный сэмпл (например, 1–2M строк) **до** разбиения, но **одинаковый для всех троих**. Зафиксируйте seed сэмплирования.

8. **Логирование.** Каждый участник должен сохранять:
   - предсказания uplift на validation и test (в parquet/CSV);
   - Qini-кривые (массивы `t`, `Q(t)`);
   - значения AUUC, Qini, Uplift@10%, Uplift@20%;
   - bootstrap CI для Qini.
   Это позволит в конце построить единый график сравнения.

9. **Не подбирайте гиперпараметры на test.** Test трогается **один раз** в самом конце. Все сравнения моделей и выбор лучшей — на validation.

10. **Response-модель как обязательный baseline.** Обучите обычный классификатор `P(Y|X)` без `T` и посчитайте его Qini. Если ваша uplift-модель не лучше response-модели, значит, она не даёт причинного сигнала.

# 7. Разбиение на трёх человек: три равных трека

Ниже — три независимых трека. Каждый участник делает **полный пайплайн**: загрузка, split, feature engineering, обучение, предсказание, метрики, Qini-кривые, сохранение артефактов.

**Общие требования для всех:**
- один и тот же split 60/20/20 (seed и индексы согласованы);
- один и тот же основной таргет — `visit` (дополнительно можно посчитать `conversion`);
- стратификация по `treatment` и `visit`;
- **ВАЖНО (из-за одного req.txt):** все используют один общий `requirements.txt`. В него добавлены `scikit-uplift`, `causalml`, `econml` и `torch` (CPU-версия);
- обязательный baseline: **response-модель** (LightGBM без `T`);
- метрики на validation: AUUC, Qini, Uplift@10%, Uplift@20%, bootstrap 95% CI для Qini;
- финальные метрики на test — один раз, после выбора лучшей модели на validation.

Треки сбалансированы по сложности. Так как у всех CPU, мы используем **сэмплирование** для тяжелых алгоритмов, чтобы каждый трек занимал не более 15–20 минут машинного времени.

## Трек A. Мета-алгоритмы на табличных моделях: T-learner, S-learner, X-learner

**Сложность:** средняя. Много аккуратной инженерии и сравнений, но алгоритмы хорошо документированы и работают на CPU мгновенно.

**Цель:** получить надёжные baseline-и и понять, насколько классические мета-алгоритмы вытягивают uplift на Criteo.

### Задачи

**A1. EDA и подготовка.**
- Загрузить датасет (или стратифицированный сэмпл 1–2M строк, согласованный с командой).
- Сделать OHE для категориальных признаков (или target encoding). Числовые — стандартизировать для логистической регрессии.
- Сохранить индексы train/val/test в CSV для всей команды.

**A2. Response-модель (baseline).**
- Обучить логистическую регрессию `P(visit|X)` без `T` на train.
- Посчитать Qini, AUUC, Uplift@10% на validation. Это нижняя планка.

**A3. T-learner и S-learner (LogReg).**
- Обучить две логистические регрессии для T-learner: \(M_1\) и \(M_0\).
- Обучить одну для S-learner с `treatment` как признаком.
- Предсказать \(\hat{\tau}(x)\) для обоих. Сравнить Qini.

**A4. T-learner и S-learner на LightGBM.**
- Повторить A3, заменив логистическую регрессию на LightGBM.
- Сравнить Qini: линейные модели vs бустинг. Оценить время обучения (LightGBM на CPU должен учиться < 2 мин).

**A5. X-learner.**
- Реализовать X-learner на LightGBM (или использовать `causalml`).
- Сравнить Qini с T-learner. Проверить, помогает ли X-learner при дисбалансе 85/15.

**A6. Анализ и артефакты.**
- Свести таблицу метрик, построить Qini-кривые на одном графике.
- Сохранить предсказания val/test в parquet.

**Что сдать:** ноутбук с кодом, таблицей, графиками; CSV с индексами split; parquet с предсказаниями.

## Трек B. Прямые uplift-методы: CVT, Uplift Random Forest, Causal Forest

**Сложность:** средняя–высокая. Нужно разобраться с библиотеками `causalml` / `econml`. Работает на CPU, но требует контроля размера выборки для деревьев.

**Цель:** проверить, дают ли прямые uplift-алгоритмы выигрыш над мета-алгоритмами за счёт встроенной оптимизации гетерогенности.

### Задачи

**B1. EDA и подготовка.**
- Использовать тот же split, что и остальные. Подготовить признаки.

**B2. Response-модель (baseline).**
- LightGBM `P(visit|X)` без `T`. Qini на validation.

**B3. Class Variable Transformation (CVT).**
- Реализовать преобразование \(Z_i\) (см. 5.5).
- Обучить логистическую регрессию и LightGBM на \(Z\).
- Пересчитать $\hat{\tau} = 2P(Z=1|X)-1$. Посчитать метрики.

**B4. Uplift Random Forest (causalml).**
- Использовать `UpliftRandomForestClassifier` из `causalml`.
- **Оптимизация:** используй `n_estimators=50`, `max_depth=5`, `min_samples_leaf=100`. Это займет 2–3 минуты на CPU.
- Сравнить Qini с CVT.

**B5. Causal Forest (econml / GRF).**
- Использовать `CausalForestDML` или `CausalForest` из `econml`.
- **КРИТИЧЕСКИ ВАЖНО:** На CPU `econml` может считаться часами, если дать ему 1M строк. **Обязательно сделай стратифицированный сэмпл 100 000 строк** из train специально для обучения этой модели.
- Обучить на сэмпле, предсказать CATE на полном validation.
- Сравнить Qini с Uplift RF.

**B6. Анализ гетерогенности.**
- Построить важность признаков для uplift (feature importance из леса).
- Выделить 2–3 сегмента с наибольшим/наименьшим uplift и проверить на validation.

**B7. Артефакты.**
- Таблица метрик, Qini-кривые, важность признаков, предсказания val/test.

**Что сдать:** ноутбук с CVT, Uplift RF (на полном train), Causal Forest (на сэмпле 100k), таблицей и графиками; parquet с предсказаниями.

## Трек C. R-learner, DR-learner и нейросетевой подход (TARNet на PyTorch)

**Сложность:** высокая, но жестко оптимизирована под  CPU (Ryzen 7 5700U), чтобы обучение заняло не более 20 минут.

**Цель:** проверить продвинутые причинные подходы (R/DR-learners) и базовую нейросеть на PyTorch. Сделать оценку неопределённости (bootstrap) и policy value.

### Задачи

**C1. EDA и подготовка.**
- Согласовать split. Для нейросети нужны стандартизированные числовые признаки и OHE категориальных.

**C2. Response-модель (baseline).**
- LightGBM `P(visit|X)` без `T`. Qini на validation.

**C3. R-learner (LightGBM).**
- Обучить $m(x) \approx E[Y|X]$ и $e(x) \approx P(T=1|X)$ на LightGBM.
- Вычислить остатки $\tilde{Y}$, $\tilde{T}$.
- Обучить финальную модель на $\tilde{Y} \approx \tau(X)\tilde{T}$ (через кастомный лосс LightGBM или взвешенную регрессию).
- Посчитать Qini. (Занимает < 2 минут на CPU).

**C4. DR-learner (LightGBM).**
- Реализовать DR-learner (doubly robust) через pseudo-outcome:

$$ \psi_i = \frac{T_i - e(X_i)}{e(X_i)(1-e(X_i))}\left(Y_i - m(X_i)\right) + m(X_i) $$

- Обучить регрессию $\tau(x)$ на $\psi_i$ с помощью LightGBM.
- Посчитать Qini.

**C5. TARNet на PyTorch (CPU-версия).**
- Реализовать простую сеть: общий MLP-ствол (2 слоя по 64 нейрона), две головы (treatment/control).
- **Оптимизация под CPU:** Возьми сэмпл **500 000 строк** из train. Используй `batch_size=4096`, `epochs=10`, `optimizer=Adam(lr=1e-3)`. На Ryzen 7 5700U это займет около 3–5 минут.
- Обучить, предсказать uplift как разность голов.
- Сравнить Qini с R-learner.

**C6. Продвинутая оценка.**
- Bootstrap Qini CI (200 ресемплов) для лучшей модели трека (ресемплинг только предсказаний на validation, без переобучения — это мгновенно).
- Policy value: оценить, сколько инкрементальных визитов даст таргетинг верхних 10% / 20% по модели, и сравнить со случайным таргетингом.

**C7. Артефакты.**
- Таблица метрик, Qini-кривые, bootstrap CI, policy value, предсказания.

**Что сдать:** ноутбук с R-learner, DR-learner, TARNet (на 500k сэмпле), bootstrap CI и policy value; parquet с предсказаниями.

## 7.1. Сводная таблица распределения задач

| Трек | Участник | Основные методы | Продвинутая часть | Вычислительная нагрузка (CPU) |
|---|---|---|---|---|
| A | 1 | T-learner, S-learner, X-learner (LogReg + LightGBM) | X-learner, сравнение линейных и бустинга | Низкая (быстро) |
| B | 2 | CVT, Uplift RF, Causal Forest | Causal Forest на сэмпле 100k | Средняя (нужен контроль памяти) |
| C | 3 | R-learner, DR-learner, TARNet (PyTorch) | Bootstrap CI, policy value, нейросеть на 500k | Высокая (до 20 минут) |

**Почему это примерно поровну:**
- У A больше моделей, но они учатся за секунды.
- У B сложнее библиотеки и интерпретация деревьев, но мы ограничили сэмпл.
- У C сложнее математика (R/DR) и нейросеть, но мы жестко зафиксировали батчи и эпохи, чтобы не выходить за рамки разумного времени.
- Все три трека включают response-baseline, EDA, split, Qini, сохранение артефактов.

# 8. Как сравнивать результаты и что сдавать

## 8.1. Единый протокол сравнения

Чтобы сравнение было честным:

1. **Один и тот же split.** Индексы train/val/test сохранены в CSV и используются всеми.
2. **Один и тот же таргет.** Основной — `visit`.
3. **Один и тот же расчёт Qini.** Лучше договориться об общей функции расчёта (например, из `scikit-uplift`), чтобы метрика считалась одинаково.
4. **Qini на validation — для выбора модели.** Test — только финальная оценка.
5. **Bootstrap CI.** Для финального сравнения Qini на test постройте 95% CI.
6. **Честность по сэмплам.** Если модель обучалась на сэмпле (Causal Forest — 100k, TARNet — 500k), это нужно указать в таблице. Сравнивать Qini нужно на **полном** validation/test.

## 8.2. Итоговая таблица (заполняется в конце)

| Трек | Модель | Обучающая выборка | AUUC (val) | Qini (val) | Uplift@10% (val) | Qini 95% CI (val) | Qini (test) |
|---|---|---|---|---|---|---|---|
| Baseline | Response LightGBM | Full train | | | | | |
| A | T-learner LogReg | Full train | | | | | |
| A | S-learner LogReg | Full train | | | | | |
| A | T-learner LightGBM | Full train | | | | | |
| A | X-learner LightGBM | Full train | | | | | |
| B | CVT LogReg | Full train | | | | | |
| B | CVT LightGBM | Full train | | | | | |
| B | Uplift RF | Full train | | | | | |
| B | Causal Forest | Sample 100k | | | | | |
| C | R-learner | Full train | | | | | |
| C | DR-learner | Full train | | | | | |
| C | TARNet (PyTorch) | Sample 500k | | | | | |

## 8.3. Что сдать в конце

Каждый участник сдаёт:
- Jupyter notebook с кодом и выводами;
- CSV/parquet с предсказаниями uplift на validation и test;
- Qini-кривые (массивы значений);
- таблицу метрик;
- краткое резюме (5–7 предложений): что сработало, что нет, какие признаки важны.

**Общий итог:** один сводный ноутбук, где построены Qini-кривые всех моделей на одном графике, сделан вывод, какой класс методов лучше работает на Criteo v2.1, и обсуждены ограничения (сэмплы, CPU).